# Templates Directory Manager

This notebook creates a directory listing of the template files, saves it to a Markdown file, and uses the information to update base.html and server.py files. This helps maintain consistency between the template directory structure and the application code that references these templates.

## Import Required Libraries

We'll import the necessary Python modules for file and directory operations.

In [ ]:
# Import Required Libraries
import os
import pathlib
from pathlib import Path
import re
import datetime

## List Files in Directory

Let's list all files in the 'src/web/templates' directory and store the results in a structured way. We'll get file names, extensions, and last modified dates.

In [ ]:
# Define the templates directory path
templates_dir = Path('src/web/templates')

# Check if directory exists
if not templates_dir.exists():
    print(f"Warning: Directory {templates_dir} does not exist.")
    print("Creating directories for demonstration purposes...")
    templates_dir.mkdir(parents=True, exist_ok=True)

# Function to list all template files
def list_template_files(directory):
    file_list = []
    
    # Walk through directory and all subdirectories
    for root, dirs, files in os.walk(directory):
        rel_path = os.path.relpath(root, directory)
        if rel_path == '.':
            rel_path = ''
        
        # Add all HTML files to list
        for file in files:
            if file.endswith('.html'):
                full_path = os.path.join(root, file)
                rel_file_path = os.path.join(rel_path, file)
                file_info = {
                    'name': file,
                    'path': rel_file_path.replace('\\', '/'),
                    'modified': datetime.datetime.fromtimestamp(os.path.getmtime(full_path)).strftime('%Y-%m-%d %H:%M:%S'),
                    'size': os.path.getsize(full_path)
                }
                file_list.append(file_info)
    
    return file_list

# Get the list of template files
template_files = list_template_files(templates_dir)

# Display the result
print(f"Found {len(template_files)} template files:")
for file in template_files:
    print(f"- {file['path']} ({file['size']} bytes, modified: {file['modified']})")

## Write Directory Listing to Markdown File

Now let's write this information to a Markdown file for documentation purposes.

In [ ]:
# Define the output Markdown file
markdown_file = Path('docs/templates_listing.md')

# Create parent directory if it doesn't exist
markdown_file.parent.mkdir(parents=True, exist_ok=True)

# Write the template listing to the Markdown file
with open(markdown_file, 'w') as f:
    f.write('# Template Directory Listing\n\n')
    f.write(f'*Generated on: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}*\n\n')
    
    f.write('## Available Templates\n\n')
    f.write('| Template Path | Size (bytes) | Last Modified |\n')
    f.write('|---------------|--------------|---------------|\n')
    
    for file in template_files:
        f.write(f"| {file['path']} | {file['size']} | {file['modified']} |\n")
    
    f.write('\n## Directory Structure\n\n')
    f.write('```\n')
    
    # Add directory structure visualization
    if templates_dir.exists():
        dirs_seen = set()
        for file in template_files:
            dir_path = os.path.dirname(file['path'])
            parts = dir_path.split('/')
            for i in range(len(parts)):
                if parts[i]:  # Skip empty parts
                    current = '/'.join(parts[:i+1])
                    if current not in dirs_seen:
                        f.write('  ' * i + '└── ' + parts[i] + '/\n')
                        dirs_seen.add(current)
            
            # Get the filename part
            filename = os.path.basename(file['path'])
            indent = len(parts) if dir_path else 0
            f.write('  ' * indent + '└── ' + filename + '\n')
    else:
        f.write("Directory structure not available - template directory doesn't exist\n")
    
    f.write('```\n')

print(f"Markdown file written to {markdown_file}")

## Update base.html

Now let's read the base.html file and update it to include the list of available templates.
We'll add a comment section that can be automatically updated by this script.

In [ ]:
# Define the base.html file path
base_html_path = Path('src/web/templates/base.html')

# Create a sample base.html if it doesn't exist
if not base_html_path.exists():
    base_html_path.parent.mkdir(parents=True, exist_ok=True)
    with open(base_html_path, 'w') as f:
        f.write('''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{% block title %}Default Title{% endblock %}</title>
    <link rel="stylesheet" href="/static/css/styles.css">
</head>
<body>
    <header>
        <nav>
            <ul>
                <li><a href="/">Home</a></li>
                <li><a href="/about">About</a></li>
                <li><a href="/contact">Contact</a></li>
            </ul>
        </nav>
    </header>

    <main>
        {% block content %}{% endblock %}
    </main>

    <footer>
        <p>&copy; 2023 ImpressionCore</p>
    </footer>

    <!-- TEMPLATE_LIST_START -->
    <!-- Available templates will be listed here -->
    <!-- TEMPLATE_LIST_END -->

</body>
</html>''')
    print(f"Created sample {base_html_path}")

# Read the base.html file
with open(base_html_path, 'r') as f:
    base_html_content = f.read()

# Create the template list HTML comment block
template_list_html = "<!-- TEMPLATE_LIST_START -->\n"
template_list_html += "<!-- Available Templates (auto-generated, do not modify):\n"
for file in template_files:
    template_list_html += f"     - {file['path']}\n"
template_list_html += "-->\n"
template_list_html += "<!-- TEMPLATE_LIST_END -->"

# Replace the template list section in base.html
pattern = r"<!-- TEMPLATE_LIST_START -->.*?<!-- TEMPLATE_LIST_END -->"
updated_base_html = re.sub(pattern, template_list_html, base_html_content, flags=re.DOTALL)

# Write the updated base.html
with open(base_html_path, 'w') as f:
    f.write(updated_base_html)

print(f"Updated {base_html_path} with template list")

## Update server.py

Finally, let's update server.py to ensure it can properly serve all the templates we've discovered. 
We'll create or modify the server.py file to include proper route handling for all templates.

In [ ]:
# Define the server.py file path
server_py_path = Path('src/web/server.py')

# Create the parent directory if it doesn't exist
server_py_path.parent.mkdir(parents=True, exist_ok=True)

# Check if server.py exists, if not create a sample one
if not server_py_path.exists():
    with open(server_py_path, 'w') as f:
        f.write('''from flask import Flask, render_template, request, redirect, url_for

app = Flask(__name__)

# Basic routes
@app.route('/')
def index():
    return render_template('index.html')

@app.route('/about')
def about():
    return render_template('about.html')

@app.route('/contact')
def contact():
    return render_template('contact.html')

# AUTO_ROUTES_START
# Additional routes will be auto-generated here
# AUTO_ROUTES_END

if __name__ == '__main__':
    app.run(debug=True)
''')
    print(f"Created sample {server_py_path}")

# Read the server.py file
with open(server_py_path, 'r') as f:
    server_py_content = f.read()

# Generate routes for all templates
auto_routes = "# AUTO_ROUTES_START\n"
auto_routes += "# Auto-generated routes based on template files (do not modify this section)\n"

# Track templates that already have standard routes defined
standard_templates = ['index.html', 'about.html', 'contact.html']
standard_routes = [t.replace('.html', '') for t in standard_templates]

# Generate routes for other templates
for file in template_files:
    # Extract the route path from the file path
    template_path = file['path']
    route_name = template_path.replace('.html', '').replace('/', '_')
    
    # Skip templates that already have standard routes
    if os.path.basename(template_path) in standard_templates:
        continue
    
    # Create a route for this template
    auto_routes += f'''
@app.route('/{route_name.replace("_", "/")}')
def {route_name}():
    return render_template('{template_path}')
'''

auto_routes += "# AUTO_ROUTES_END"

# Replace the auto routes section in server.py
pattern = r"# AUTO_ROUTES_START.*?# AUTO_ROUTES_END"
updated_server_py = re.sub(pattern, auto_routes, server_py_content, flags=re.DOTALL)

# Write the updated server.py
with open(server_py_path, 'w') as f:
    f.write(updated_server_py)

print(f"Updated {server_py_path} with routes for all templates")

## Summary

This notebook has successfully:

1. Listed all template files in the src/web/templates directory
2. Written a structured Markdown documentation file with the template listing
3. Updated base.html with a comment section listing all available templates
4. Modified server.py to include routes for all template files

These automated updates ensure consistency between the template directory structure and the application code. Run this notebook whenever you add, remove, or modify template files to keep your documentation and routes up-to-date.

## Next Steps

To further enhance this script:

1. Add error handling for file operations
2. Implement a backup mechanism for modified files
3. Create a command-line version that could be run as part of a CI/CD pipeline
4. Add template validation to check for common errors
5. Generate an HTML report of template usage statistics

# Templates Directory Manager

This notebook creates a directory listing of the `src/web/templates` folder, saves it to a Markdown file, and demonstrates how to update base.html and server.py with this information.

## Import Required Libraries

We'll need libraries for file operations and path handling.

In [ ]:
# Import required libraries for file system operations
import os
import pathlib
from datetime import datetime
import json

## List Files in Directory

Let's get all the template files from the src/web/templates directory.

In [ ]:
# Define the path to templates directory
templates_dir = pathlib.Path("src/web/templates")

# Check if the directory exists, if not create it for demonstration
if not templates_dir.exists():
    print(f"Directory {templates_dir} does not exist. Creating it for demonstration.")
    templates_dir.mkdir(parents=True, exist_ok=True)
    
    # Create some sample templates files for demonstration
    (templates_dir / "home.html").touch()
    (templates_dir / "about.html").touch()
    (templates_dir / "contact.html").touch()
    (templates_dir / "user_profile.html").touch()

# Get all HTML files in the directory
template_files = list(templates_dir.glob("*.html"))

# Display the files found
print(f"Found {len(template_files)} template files:")
for file in template_files:
    print(f" - {file.name}")

## Write Directory Listing to Markdown File

Now we'll create a Markdown file with the directory listing, including file sizes and last modified dates.

In [ ]:
# Create the markdown content
markdown_content = f"# Templates Directory Listing\n\n"
markdown_content += f"Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
markdown_content += "| File Name | Size (KB) | Last Modified |\n"
markdown_content += "|-----------|-----------|---------------|\n"

# Add each file to the table
for file in template_files:
    file_stat = file.stat()
    size_kb = file_stat.st_size / 1024
    last_modified = datetime.fromtimestamp(file_stat.st_mtime).strftime('%Y-%m-%d %H:%M:%S')
    markdown_content += f"| {file.name} | {size_kb:.2f} | {last_modified} |\n"

# Path for markdown file
md_file_path = pathlib.Path("src/web/templates_listing.md")

# Write to markdown file
with open(md_file_path, 'w') as md_file:
    md_file.write(markdown_content)

print(f"Markdown file created at: {md_file_path}")
print("Content preview:")
print(markdown_content[:500] + "..." if len(markdown_content) > 500 else markdown_content)

## Update base.html

Now we'll demonstrate how to read the generated Markdown file and update base.html with a navigation menu using the templates list.

In [ ]:
# Create a sample base.html template if it doesn't exist
base_html_path = templates_dir / "base.html"

# Sample base.html content
base_html_content = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{% block title %}Default Title{% endblock %}</title>
    <link rel="stylesheet" href="/static/css/style.css">
</head>
<body>
    <header>
        <nav>
            <ul>
                <!-- TEMPLATE_NAVIGATION_PLACEHOLDER -->
            </ul>
        </nav>
    </header>
    <main>
        {% block content %}
        {% endblock %}
    </main>
    <footer>
        <p>&copy; {% now 'Y' %} ImpressionCore</p>
    </footer>
</body>
</html>"""

# Write the sample base.html if it doesn't exist
if not base_html_path.exists():
    with open(base_html_path, 'w') as f:
        f.write(base_html_content)
    print(f"Created sample base.html at {base_html_path}")

# Read the current base.html content
with open(base_html_path, 'r') as f:
    base_content = f.read()

# Generate navigation items from template files
nav_items = ""
for file in template_files:
    page_name = file.stem
    # Skip base.html in navigation
    if page_name != "base":
        display_name = page_name.replace("_", " ").title()
        nav_items += f'                <li><a href="/{page_name}">{display_name}</a></li>\n'

# Update the base.html with navigation items
updated_base_content = base_content.replace("<!-- TEMPLATE_NAVIGATION_PLACEHOLDER -->", nav_items)

# Write the updated base.html
with open(base_html_path, 'w') as f:
    f.write(updated_base_content)

print(f"Updated base.html with navigation menu")
print("Navigation items added:")
print(nav_items)

## Update server.py

Finally, we'll demonstrate how to update server.py to dynamically load and serve templates based on our listing.

In [ ]:
# Create a simple server.py file if it doesn't exist
server_py_path = pathlib.Path("src/web/server.py")

# Create the directory if it doesn't exist
server_py_path.parent.mkdir(parents=True, exist_ok=True)

# Sample server.py content
server_py_content = """from flask import Flask, render_template
import os
import pathlib

app = Flask(__name__)

# DYNAMIC_ROUTES_PLACEHOLDER

@app.route('/')
def home():
    return render_template('home.html')

if __name__ == '__main__':
    app.run(debug=True)
"""

# Write the sample server.py if it doesn't exist
if not server_py_path.exists():
    with open(server_py_path, 'w') as f:
        f.write(server_py_content)
    print(f"Created sample server.py at {server_py_path}")

# Read the current server.py content
with open(server_py_path, 'r') as f:
    server_content = f.read()

# Generate dynamic route definitions
route_definitions = "# Dynamically generated routes\n"
route_definitions += "# Last updated: " + datetime.now().strftime('%Y-%m-%d %H:%M:%S') + "\n\n"

# Create a dictionary to store template metadata
templates_meta = []

for file in template_files:
    page_name = file.stem
    if page_name != "base" and page_name != "home":  # Skip base.html and home.html (already defined)
        display_name = page_name.replace("_", " ").title()
        route_path = f"/{page_name}"
        
        # Add template to metadata
        templates_meta.append({
            "name": page_name,
            "display_name": display_name,
            "path": route_path,
            "file": file.name
        })
        
        # Create route function
        route_definitions += f"""@app.route('{route_path}')
def {page_name.replace('-', '_')}():
    return render_template('{file.name}')\n\n"""

# Save template metadata to JSON for future reference
templates_meta_path = pathlib.Path("src/web/templates_meta.json")
with open(templates_meta_path, 'w') as f:
    json.dump(templates_meta, f, indent=2)

# Update server.py with dynamic route definitions
updated_server_content = server_content.replace("# DYNAMIC_ROUTES_PLACEHOLDER", route_definitions)

# Write the updated server.py
with open(server_py_path, 'w') as f:
    f.write(updated_server_content)

print(f"Updated server.py with dynamic routes")
print(f"Template metadata saved to {templates_meta_path}")
print("Dynamic routes added:")
print(route_definitions[:500] + "..." if len(route_definitions) > 500 else route_definitions)

## Summary

In this notebook, we have:

1. Listed all HTML template files in the src/web/templates directory
2. Created a Markdown file (templates_listing.md) with the directory listing
3. Updated base.html with navigation links for all templates
4. Modified server.py to dynamically serve all templates
5. Created a JSON metadata file for additional template information

This process can be run periodically to keep the website navigation and routing updated whenever new templates are added.

# Templates Directory Management

This notebook helps manage the templates directory by:
1. Creating a directory listing from 'src/web/templates'
2. Saving the listing to templates_listing.md
3. Using the listing to update base.html and server.py files

## Import Required Libraries

In [ ]:
import os
import re
from pathlib import Path
import datetime

## List Files in 'src/web/templates' Directory

We'll create a function to recursively list all templates in the directory and track their relationships.

In [ ]:
def list_template_directory(templates_dir='src/web/templates'):
    """
    List all files in the templates directory and return them in a structured format.
    
    Args:
        templates_dir (str): Path to the templates directory
        
    Returns:
        dict: A dictionary with file paths and their metadata
    """
    templates = {}
    
    # Check if directory exists
    if not os.path.exists(templates_dir):
        print(f"Warning: Directory {templates_dir} does not exist!")
        return templates
    
    # Walk through the directory
    for root, dirs, files in os.walk(templates_dir):
        for file in files:
            if file.endswith('.html') or file.endswith('.j2'):
                file_path = os.path.join(root, file)
                relative_path = os.path.relpath(file_path, templates_dir)
                
                # Read file to extract relevant information
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read()
                
                # Extract extends relationships (for Jinja templates)
                extends_match = re.search(r'{%\s*extends\s+[\'"](.+?)[\'"]\s*%}', content)
                extends = extends_match.group(1) if extends_match else None
                
                # Extract included templates
                includes = re.findall(r'{%\s*include\s+[\'"](.+?)[\'"]\s*%}', content)
                
                # Get file metadata
                stat = os.stat(file_path)
                
                templates[relative_path] = {
                    'extends': extends,
                    'includes': includes,
                    'size': stat.st_size,
                    'modified': datetime.datetime.fromtimestamp(stat.st_mtime).strftime('%Y-%m-%d %H:%M:%S'),
                    'path': file_path
                }
    
    return templates

In [ ]:
# Execute the function to list all templates
templates = list_template_directory()

print(f"Found {len(templates)} templates")
for template, info in list(templates.items())[:5]:  # Show first 5 for preview
    print(f"- {template}")
    if info['extends']:
        print(f"  Extends: {info['extends']}")
    if info['includes']:
        print(f"  Includes: {', '.join(info['includes'])}")

## Write Directory Listing to templates_listing.md

Now we'll format the template information and write it to a Markdown file for documentation purposes.

In [ ]:
def write_template_listing(templates, output_file='templates_listing.md'):
    """
    Write the templates listing to a markdown file.
    
    Args:
        templates (dict): Dictionary of templates and their metadata
        output_file (str): Path to the output markdown file
    """
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write("# Templates Directory Listing\n\n")
        f.write(f"*Generated on: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*\n\n")
        
        # Write summary
        f.write("## Summary\n\n")
        f.write(f"Total Templates: **{len(templates)}**\n\n")
        
        # Create inheritance tree
        f.write("## Template Inheritance\n\n")
        f.write("```\n")
        
        # Find base templates (those that don't extend others)
        base_templates = [t for t, info in templates.items() if not info['extends']]
        
        def print_tree(template, indent=0):
            f.write("  " * indent + template + "\n")
            children = [t for t, info in templates.items() if info['extends'] == template]
            for child in sorted(children):
                print_tree(child, indent + 1)
        
        for base in sorted(base_templates):
            print_tree(base)
        
        f.write("```\n\n")
        
        # Write detailed listing
        f.write("## Detailed Listing\n\n")
        
        for template_path in sorted(templates.keys()):
            info = templates[template_path]
            f.write(f"### {template_path}\n\n")
            f.write(f"- **Size**: {info['size']} bytes\n")
            f.write(f"- **Last Modified**: {info['modified']}\n")
            
            if info['extends']:
                f.write(f"- **Extends**: `{info['extends']}`\n")
            
            if info['includes']:
                f.write("- **Includes**:\n")
                for include in info['includes']:
                    f.write(f"  - `{include}`\n")
            
            f.write("\n")
    
    print(f"Template listing written to {output_file}")

In [ ]:
# Write the template listing to a file
write_template_listing(templates)

## Update base.html

Next, we'll update the base.html template to include information about available templates.
This helps developers see what templates are available directly from the base template.

In [ ]:
def update_base_html(templates, base_html_path='src/web/templates/base.html'):
    """
    Update the base.html file with template information.
    
    Args:
        templates (dict): Dictionary of templates and their metadata
        base_html_path (str): Path to the base.html file
    """
    # Check if base.html exists
    if not os.path.exists(base_html_path):
        print(f"Warning: {base_html_path} does not exist!")
        return False
    
    # Read the current content
    with open(base_html_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Define the template information to insert
    template_info = """
    {# Template information - auto-generated #}
    {# Available templates:
    {% for template in available_templates %}
    * {{ template }}
    {% endfor %}
    #}
    """
    
    # Format the template info with actual template paths
    template_paths = sorted(templates.keys())
    template_paths_str = '\n    * '.join(template_paths)
    formatted_info = template_info.replace("{% for template in available_templates %}\n    * {{ template }}\n    {% endfor %}", 
                                          f"{template_paths_str}")
    
    # Check if there's already template information block
    template_info_pattern = r"\{#\s*Template information - auto-generated[\s\S]+?#\}"
    if re.search(template_info_pattern, content):
        # Replace existing template information
        updated_content = re.sub(template_info_pattern, formatted_info.strip(), content)
    else:
        # Insert at the beginning of the file
        updated_content = formatted_info + "\n" + content
    
    # Write the updated content back
    with open(base_html_path, 'w', encoding='utf-8') as f:
        f.write(updated_content)
    
    print(f"Updated {base_html_path} with template information")
    return True

In [ ]:
# Update the base.html file
try:
    update_base_html(templates)
except Exception as e:
    print(f"Error updating base.html: {e}")

## Update server.py

Finally, we'll update server.py to ensure it has routes for all the templates
and properly handles template relationships.

In [ ]:
def update_server_py(templates, server_py_path='src/web/server.py'):
    """
    Update server.py file with template routes.
    
    Args:
        templates (dict): Dictionary of templates and their metadata
        server_py_path (str): Path to the server.py file
    """
    # Check if server.py exists
    if not os.path.exists(server_py_path):
        print(f"Warning: {server_py_path} does not exist!")
        return False
    
    # Read the current content
    with open(server_py_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Create template route definitions
    template_routes = []
    
    for template_path in sorted(templates.keys()):
        # Skip include files and partials
        if template_path.startswith('_') or template_path.startswith('includes/') or template_path.startswith('partials/'):
            continue
        
        route_path = template_path.replace('.html', '').replace('.j2', '')
        if route_path.endswith('index'):
            route_path = route_path[:-5] if route_path != 'index' else '/'
        else:
            route_path = '/' + route_path
        
        template_routes.append(f"""
@app.route('{route_path}')
def render_{route_path.replace('/', '_').replace('-', '_').strip('_')}():
    return render_template('{template_path}')
""")
    
    # Check if there's already an auto-generated routes section
    routes_pattern = r"# Auto-generated routes for templates[\s\S]+?# End of auto-generated routes"
    routes_block = "# Auto-generated routes for templates\n" + ''.join(template_routes) + "\n# End of auto-generated routes"
    
    if re.search(routes_pattern, content):
        # Replace existing routes section
        updated_content = re.sub(routes_pattern, routes_block, content)
    else:
        # Find a good place to insert routes (after app definition)
        app_pattern = r"app\s*=\s*Flask\(__name__.*?\)"
        if re.search(app_pattern, content):
            updated_content = re.sub(app_pattern, lambda m: m.group(0) + "\n\n" + routes_block, content)
        else:
            # If we can't find the app definition, just append to the end
            updated_content = content + "\n\n" + routes_block
    
    # Write the updated content back
    with open(server_py_path, 'w', encoding='utf-8') as f:
        f.write(updated_content)
    
    print(f"Updated {server_py_path} with {len(template_routes)} template routes")
    return True

In [ ]:
# Update the server.py file
try:
    update_server_py(templates)
except Exception as e:
    print(f"Error updating server.py: {e}")

## Summary

This notebook has:
1. Listed all templates in the src/web/templates directory
2. Created a comprehensive templates_listing.md file documenting the templates
3. Updated base.html with template information for developers
4. Updated server.py with routes for all available templates

These steps help maintain consistency between the templates directory structure and the application code.

# Templates Directory Manager

This notebook helps manage the templates directory by:
1. Creating a directory listing
2. Saving the listing to a Markdown file
3. Updating base.html with template references
4. Modifying server.py to dynamically load templates

This approach follows the structured repository guidelines and applies modular code organization principles.

## Import Required Libraries

We'll import necessary libraries for directory operations, path handling, and file manipulation.

In [ ]:
import os
import pathlib
from datetime import datetime
import re
import shutil
from typing import List, Dict, Optional

## List Files in Directory

Now we'll scan the templates directory and organize the files by type.

In [ ]:
# Define the templates directory path
TEMPLATES_DIR = "src/web/templates"

def get_template_files(directory: str = TEMPLATES_DIR) -> Dict[str, List[str]]:
    """
    Get all template files organized by extension.
    
    Args:
        directory: The directory to scan
        
    Returns:
        Dictionary with extensions as keys and lists of filenames as values
    """
    # Ensure directory exists
    if not os.path.exists(directory):
        print(f"Warning: Directory '{directory}' does not exist")
        return {}
        
    # Scan directory and organize files by extension
    files_by_type = {}
    
    for file in os.listdir(directory):
        file_path = os.path.join(directory, file)
        
        # Skip directories
        if os.path.isdir(file_path):
            continue
            
        # Get file extension
        _, ext = os.path.splitext(file)
        ext = ext.lower()
        
        # Add to dictionary
        if ext not in files_by_type:
            files_by_type[ext] = []
        files_by_type[ext].append(file)
    
    # Sort files within each category
    for ext in files_by_type:
        files_by_type[ext].sort()
        
    return files_by_type

# Get template files
template_files = get_template_files()
print(f"Found {sum(len(files) for files in template_files.values())} template files")

# Display files by type
for ext, files in template_files.items():
    print(f"\n{ext} files ({len(files)}):")
    for file in files:
        print(f"  - {file}")

## Write Directory Listing to Markdown File

Now we'll create a well-formatted Markdown file with the template listing.

In [ ]:
def write_templates_listing_md(files_by_type: Dict[str, List[str]], output_file: str = "templates_listing.md") -> str:
    """
    Write template files listing to a markdown file.
    
    Args:
        files_by_type: Dictionary with extensions as keys and lists of filenames as values
        output_file: Path to the output markdown file
        
    Returns:
        Path to the created file
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    with open(output_file, "w") as f:
        f.write(f"# Templates Directory Listing\n\n")
        f.write(f"Generated on: {timestamp}\n\n")
        
        # Summary section
        total_files = sum(len(files) for files in files_by_type.values())
        f.write(f"## Summary\n\n")
        f.write(f"Total files: {total_files}\n\n")
        
        for ext, files in files_by_type.items():
            f.write(f"- {ext} files: {len(files)}\n")
        
        # Detailed listings by type
        f.write(f"\n## Files by Type\n\n")
        
        for ext, files in files_by_type.items():
            ext_name = ext.strip('.') if ext else 'no_extension'
            f.write(f"### {ext_name.upper()} Files\n\n")
            
            for file in files:
                f.write(f"- `{file}`\n")
            
            f.write("\n")
    
    print(f"Template listing saved to {output_file}")
    return output_file

# Write the templates listing to a markdown file
listing_file = write_templates_listing_md(template_files)

# Display the content of the generated markdown file
with open(listing_file, "r") as f:
    print(f.read())

## Update base.html

Now we'll demonstrate how to update base.html to include references to the templates. This will:

1. Read the existing base.html file
2. Add a templates menu section that lists available templates
3. Write the updated content back to base.html

In [ ]:
def update_base_html(files_by_type: Dict[str, List[str]], base_html_path: str = "src/web/templates/base.html") -> bool:
    """
    Update the base.html file with template references.
    
    Args:
        files_by_type: Dictionary with extensions as keys and lists of filenames as values
        base_html_path: Path to the base.html file
        
    Returns:
        True if successful, False otherwise
    """
    # Check if base.html exists
    if not os.path.exists(base_html_path):
        print(f"Error: {base_html_path} does not exist")
        return False
        
    # Create a backup
    backup_path = f"{base_html_path}.backup"
    shutil.copy2(base_html_path, backup_path)
    print(f"Created backup of base.html at {backup_path}")
    
    # Read the base.html content
    with open(base_html_path, "r") as f:
        content = f.read()
    
    # Generate templates menu HTML
    template_menu = """
    <!-- Templates Menu - Auto-generated -->
    <div class="templates-menu">
        <h3>Available Templates</h3>
        <ul class="templates-list">
    """
    
    # Add HTML templates first (most important)
    if ".html" in files_by_type:
        template_menu += '        <li class="template-category">HTML Templates</li>\n'
        for file in files_by_type[".html"]:
            if file != "base.html":  # Skip base.html itself
                template_name = file.replace(".html", "")
                template_menu += f'        <li><a href="/templates/{template_name}">{template_name}</a></li>\n'
    
    # Add other template types
    for ext, files in files_by_type.items():
        if ext != ".html" and files:  # Skip HTML (already processed) and empty lists
            ext_name = ext.strip('.').upper()
            template_menu += f'        <li class="template-category">{ext_name} Files</li>\n'
            for file in files:
                template_menu += f'        <li><a href="/static/templates/{file}">{file}</a></li>\n'
    
    template_menu += """
        </ul>
    </div>
    <!-- End Templates Menu -->
    """
    
    # Check if templates menu already exists
    if "<!-- Templates Menu - Auto-generated -->" in content:
        # Replace existing menu
        content = re.sub(
            r"<!-- Templates Menu - Auto-generated -->.*?<!-- End Templates Menu -->",
            template_menu,
            content,
            flags=re.DOTALL
        )
    else:
        # Add menu before the closing </body> tag
        content = content.replace("</body>", f"{template_menu}\n</body>")
    
    # Write updated content
    with open(base_html_path, "w") as f:
        f.write(content)
    
    print(f"Updated {base_html_path} with templates menu")
    return True

# Demo the update function (won't actually write if file doesn't exist)
try:
    update_base_html(template_files)
except Exception as e:
    print(f"Demo mode: Would update base.html with template references. Error: {e}")

# Show a sample of what would be added to base.html
html_templates = template_files.get(".html", [])
print("\nSample template menu code that would be added to base.html:")
print("""
<!-- Templates Menu - Auto-generated -->
<div class="templates-menu">
    <h3>Available Templates</h3>
    <ul class="templates-list">""")

if html_templates:
    print('        <li class="template-category">HTML Templates</li>')
    for file in html_templates[:3]:  # Show up to 3 examples
        if file != "base.html":
            template_name = file.replace(".html", "")
            print(f'        <li><a href="/templates/{template_name}">{template_name}</a></li>')
    if len(html_templates) > 3:
        print("        ... (more templates) ...")

print("""    </ul>
</div>
<!-- End Templates Menu -->
""")

## Update server.py

Finally, we'll demonstrate how to modify server.py to dynamically load templates based on the directory listing.

In [ ]:
def update_server_py(files_by_type: Dict[str, List[str]], server_py_path: str = "src/web/server.py") -> bool:
    """
    Update server.py to dynamically load templates.
    
    Args:
        files_by_type: Dictionary with extensions as keys and lists of filenames as values
        server_py_path: Path to the server.py file
        
    Returns:
        True if successful, False otherwise
    """
    # Check if server.py exists
    if not os.path.exists(server_py_path):
        print(f"Error: {server_py_path} does not exist")
        return False
        
    # Create a backup
    backup_path = f"{server_py_path}.backup"
    shutil.copy2(server_py_path, backup_path)
    print(f"Created backup of server.py at {backup_path}")
    
    # Read the server.py content
    with open(server_py_path, "r") as f:
        content = f.read()
    
    # Generate dynamic template routes code
    dynamic_routes = """
# Dynamic template routes - Auto-generated
@app.route('/templates/<template_name>')
def serve_template(template_name):
    \"\"\"Dynamically serve templates.\"\"\"
    # Add .html extension if not provided
    if not template_name.endswith('.html'):
        template_name += '.html'
    
    # Check if template exists
    template_path = os.path.join('templates', template_name)
    if not os.path.exists(template_path):
        return render_template('error.html', error=f"Template {template_name} not found"), 404
    
    # Render the template with optional context
    return render_template(template_name)

# Available templates dictionary - Auto-generated
AVAILABLE_TEMPLATES = {
"""
    
    # Add HTML templates to the AVAILABLE_TEMPLATES dictionary
    if ".html" in files_by_type:
        for file in files_by_type[".html"]:
            template_name = file.replace(".html", "")
            dynamic_routes += f"    '{template_name}': '{file}',\n"
    
    dynamic_routes += """
}

@app.route('/api/templates')
def list_templates():
    \"\"\"Return a list of available templates.\"\"\"
    return jsonify(AVAILABLE_TEMPLATES)
# End of dynamic template routes
"""
    
    # Check if dynamic routes already exist
    if "# Dynamic template routes - Auto-generated" in content:
        # Replace existing dynamic routes
        content = re.sub(
            r"# Dynamic template routes - Auto-generated.*?# End of dynamic template routes",
            dynamic_routes,
            content,
            flags=re.DOTALL
        )
    else:
        # Add dynamic routes before the app.run() line
        if "app.run" in content:
            content = content.replace("app.run", f"{dynamic_routes}\napp.run")
        else:
            # Append to end if app.run isn't found
            content += f"\n{dynamic_routes}\n"
    
    # Write updated content
    with open(server_py_path, "w") as f:
        f.write(content)
    
    print(f"Updated {server_py_path} with dynamic template routes")
    return True

# Demo the update function (won't actually write if file doesn't exist)
try:
    update_server_py(template_files)
except Exception as e:
    print(f"Demo mode: Would update server.py with dynamic template handling. Error: {e}")

# Show a sample of what would be added to server.py
html_templates = template_files.get(".html", [])
print("\nSample code that would be added to server.py:")
print("""
# Dynamic template routes - Auto-generated
@app.route('/templates/<template_name>')
def serve_template(template_name):
    """Dynamically serve templates."""
    # Add .html extension if not provided
    if not template_name.endswith('.html'):
        template_name += '.html'
    
    # Check if template exists
    template_path = os.path.join('templates', template_name)
    if not os.path.exists(template_path):
        return render_template('error.html', error=f"Template {template_name} not found"), 404
    
    # Render the template with optional context
    return render_template(template_name)

# Available templates dictionary - Auto-generated
AVAILABLE_TEMPLATES = {""")

if html_templates:
    for file in html_templates[:3]:  # Show up to 3 examples
        template_name = file.replace(".html", "")
        print(f"    '{template_name}': '{file}',")
    if len(html_templates) > 3:
        print("    # ... more templates ...")

print("""}

@app.route('/api/templates')
def list_templates():
    """Return a list of available templates."""
    return jsonify(AVAILABLE_TEMPLATES)
# End of dynamic template routes
""")

## Create a Main Function to Run All Steps

Let's create a main function that can run all the above steps in sequence.

In [ ]:
def main(templates_dir: str = "src/web/templates",
         output_md: str = "templates_listing.md",
         base_html_path: str = "src/web/templates/base.html",
         server_py_path: str = "src/web/server.py") -> None:
    """
    Run all directory management steps in sequence.
    
    Args:
        templates_dir: Path to templates directory
        output_md: Path to output markdown file
        base_html_path: Path to base.html file
        server_py_path: Path to server.py file
    """
    print(f"Starting templates directory management process")
    print(f"Templates directory: {templates_dir}")
    
    # 1. Get template files
    print("\n1. Getting template files...")
    try:
        template_files = get_template_files(templates_dir)
        total_files = sum(len(files) for files in template_files.values())
        print(f"✓ Found {total_files} template files")
    except Exception as e:
        print(f"✗ Error getting template files: {e}")
        return
    
    # 2. Write templates listing
    print("\n2. Writing templates listing to markdown...")
    try:
        listing_file = write_templates_listing_md(template_files, output_md)
        print(f"✓ Templates listing saved to {listing_file}")
    except Exception as e:
        print(f"✗ Error writing templates listing: {e}")
        return
    
    # 3. Update base.html
    print("\n3. Updating base.html...")
    try:
        if os.path.exists(base_html_path):
            result = update_base_html(template_files, base_html_path)
            if result:
                print(f"✓ Updated {base_html_path} successfully")
            else:
                print(f"✗ Failed to update {base_html_path}")
        else:
            print(f"! {base_html_path} does not exist, skipping update")
    except Exception as e:
        print(f"✗ Error updating base.html: {e}")
    
    # 4. Update server.py
    print("\n4. Updating server.py...")
    try:
        if os.path.exists(server_py_path):
            result = update_server_py(template_files, server_py_path)
            if result:
                print(f"✓ Updated {server_py_path} successfully")
            else:
                print(f"✗ Failed to update {server_py_path}")
        else:
            print(f"! {server_py_path} does not exist, skipping update")
    except Exception as e:
        print(f"✗ Error updating server.py: {e}")
    
    print("\nTemplate management process complete!")

# Run the main function (in demo mode)
print("Demo run of the main function - will not modify files if they don't exist")
main()

## Conclusion

This notebook provides a comprehensive solution for managing templates in a web application:

1. We've created a system to scan and inventory template files
2. Generated a well-formatted Markdown file with the template listing
3. Demonstrated how to update base.html with template references
4. Shown how to modify server.py to dynamically load templates

To use this in production:
1. Run the `main()` function with the correct paths for your project
2. Verify the generated Markdown file for accuracy
3. Check that base.html and server.py were updated correctly
4. Test the dynamic template loading functionality

This approach follows best practices for modular code organization and documentation.